# dieWolke: small Kharif demo (Colab)

Runs the models trained on SEN12MS-CR (spring, 2,434 patches) on a **small Indian Kharif
benchmark**: real July–October Sentinel-2 cropland patches from one district, with real monsoon
clouds and shadows pasted onto clear patches so the true image is known. Nothing is trained here.

| Step | What | Time |
|---|---|---|
| 1 | Setup | 2 min |
| 2 | Settings (fill in your Earth Engine project) | – |
| 3 | Find July–Oct scenes over the district | 1–3 min |
| 4 | Download patches from Earth Engine | 5–20 min |
| 5 | Build the benchmark | 1–2 min |
| 6 | Get the trained models from Kaggle | 2–5 min |
| 7 | Run the models and score them | 2–5 min |
| 8 | Before/after pictures | 1 min |
| 9 | Downstream land-cover classification | 2–5 min |

Runtime: **GPU (T4)** is faster, CPU also works at this size. Outputs go to Google Drive
(`MyDrive/dieWolke/kharif_demo_*`), so a disconnect only loses the current step.

**Caveats for presenting the results**
- The models never saw India or the monsoon (trained on 2,434 spring SEN12MS-CR patches).
- The classification labels are ESA WorldCover land-cover classes (cropland, trees, built-up, …),
  a proxy, **not crop types**.
- One district and one or two seasons: a demo, not a benchmark result.

## 1. Setup

In [ ]:
import os, glob, shutil, json
REPO_URL, REPO_BRANCH = 'https://github.com/Rasvanth28/DieWolke.git', 'shrey'
CODE = '/content/DieWolke'
os.chdir('/content')
if os.path.exists(CODE):
    shutil.rmtree(CODE)
!git clone -q --depth 1 -b {REPO_BRANCH} {REPO_URL} {CODE}
%cd {CODE}
!pip install -q -e ".[clouds,kharif]"
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/dieWolke'
os.makedirs(DRIVE, exist_ok=True)

## 2. Settings

Put your Earth Engine project ID in `EE_PROJECT` (the Google Cloud project registered for Earth
Engine, shown at https://code.earthengine.google.com under your account).

Ludhiana (Punjab) is a paddy-dominated district. Late October is usually clear after the monsoon,
so one year often gives enough clear patches; step 3 tells you. If it reports too few clear
scenes, add a year (e.g. `[2021, 2022]`) or a second district (`'karnal'`, `'sangrur'`) and rerun
from step 3.

In [ ]:
EE_PROJECT = ''                 # <- your Earth Engine / Google Cloud project ID
AOIS = ['ludhiana']             # district names from kharif/kharif.yaml: ludhiana sangrur karnal raipur bardhaman
YEARS = [2022]
MAX_SCENES, MAX_CLEAR, MAX_DONOR = 6, 10, 10   # per district-year / per scene; small for the demo

assert EE_PROJECT, 'set EE_PROJECT first'
import yaml
cfg = yaml.safe_load(open('kharif/kharif.yaml'))
cfg.update(project=EE_PROJECT, years=YEARS,
           raw_dir=f'{DRIVE}/kharif_demo_raw', benchmark_dir=f'{DRIVE}/kharif_demo_benchmark',
           max_scenes_per_aoi_year=MAX_SCENES, max_clear_per_scene=MAX_CLEAR, max_donor_per_scene=MAX_DONOR)
cfg['aois'] = [a for a in cfg['aois'] if a['name'] in AOIS]
assert cfg['aois'], f'none of {AOIS} is in kharif/kharif.yaml'
yaml.safe_dump(cfg, open('kharif/demo.yaml', 'w'), sort_keys=False)
RAW, BENCH = cfg['raw_dir'], cfg['benchmark_dir']
print(open('kharif/demo.yaml').read())

## 3. Find Kharif scenes

The first cell opens a Google sign-in for Earth Engine (allow it). The scripts then reuse the
saved login.

In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project=EE_PROJECT)
print('Earth Engine OK:', ee.String('hello').getInfo())

In [ ]:
!python kharif/find_scenes.py --config kharif/demo.yaml

## 4. Download patches

The dry run only counts how many clear cropland patches and cloud-donor patches exist. You need
at least a few of each (ideally 10+ clear). If there are too few, go back to step 2 and add a
year or district.

In [ ]:
!python kharif/export_patches.py --config kharif/demo.yaml --dry_run

In [ ]:
!python kharif/export_patches.py --config kharif/demo.yaml
!wc -l {RAW}/manifest.jsonl

## 5. Build the benchmark

In [ ]:
!python kharif/build_benchmark.py --config kharif/demo.yaml
!ls -la {BENCH}
SPLITS = [s for s in ('kharif_synth', 'kharif_real') if os.path.exists(f'{BENCH}/{s}_clear.npy')]
print('splits:', SPLITS)

## 6. Get the trained models

Only `best.pt` of two runs is needed (about 130 MB each; each file contains its own config).

**Option A (simplest): download by hand.** On Kaggle, open the notebook that trained each run,
go to its **Output** tab and download:
- `runs/full/best.pt` (from your training notebook, e.g. `diewolke-train`)
- `runs/matched_baseline/best.pt` (from the notebook that trained the matched baseline)

Upload them to Google Drive as `MyDrive/dieWolke/models/full/best.pt` and
`MyDrive/dieWolke/models/matched_baseline/best.pt`.

**Option B: Kaggle API.** Add Colab secrets `KAGGLE_USERNAME` and `KAGGLE_KEY` (Kaggle →
Settings → API → Create New Token) and set the two notebook slugs below; the cell downloads
the notebook outputs and keeps the right `best.pt` files.

In [ ]:
MODELS = f'{DRIVE}/models'
RUNS = {'full': 'kcrasvanth/diewolke-train',                    # Kaggle notebook whose output has runs/full
        'matched_baseline': 'kcrasvanth/notebook2133efb9e0'}    # ... and runs/matched_baseline
missing = [r for r in RUNS if not os.path.exists(f'{MODELS}/{r}/best.pt')]
if missing:  # option B
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    !pip install -q kaggle
    for r in missing:
        tmp = f'/content/kaggle_out/{r}'
        !kaggle kernels output {RUNS[r]} -p {tmp} -q
        hits = sorted(glob.glob(f'{tmp}/**/runs/{r}/best.pt', recursive=True))
        assert hits, f'no runs/{r}/best.pt in the output of {RUNS[r]}'
        os.makedirs(f'{MODELS}/{r}', exist_ok=True)
        shutil.copy(hits[0], f'{MODELS}/{r}/best.pt')
        shutil.rmtree(tmp)
!ls -la {MODELS}/*/best.pt

## 7. Run the models and score them

`kharif_synth` has exact ground truth and exact masks (real monsoon clouds pasted onto clear
patches). `kharif_real` (only if the district had cloudy/clear pairs within 30 days) uses the
real cloudy image and a different-date clear image, like SEN12MS-CR.

In [ ]:
EVAL = f'{DRIVE}/kharif_demo_eval'
for s in SPLITS:
    !python -m diewolke.evaluate --runs {MODELS}/full/best.pt {MODELS}/matched_baseline/best.pt --cache {BENCH} --split {s} --calibrate_on "" --save_predictions --out {EVAL}/{s}
    print(open(f'{EVAL}/{s}/report.md').read())

## 8. Before/after pictures

Rows are patches. Columns: SAR, cloudy input, mask, our reconstruction, true clear image and,
for the full model, the uncertainty map (bright = less sure).

In [ ]:
from IPython.display import Image, display
S = 'kharif_synth'
import numpy as np
N = len(np.load(f'{BENCH}/{S}_clear.npy', mmap_mode='r'))
print(N, 'patches in', S)
for start in range(0, min(N, 8), 4):   # up to 8 patches, 4 per figure
    out = f'{EVAL}/{S}/recon_full_{start}.png'
    !python scripts/visualize_samples.py --cache {BENCH} --split {S} --pred {EVAL}/{S}/pred_full.npy --scale {EVAL}/{S}/scale_full.npy --n 4 --start {start} --out {out}
    display(Image(out))
out = f'{EVAL}/{S}/recon_matched_0.png'
!python scripts/visualize_samples.py --cache {BENCH} --split {S} --pred {EVAL}/{S}/pred_matched_baseline.npy --n 4 --start 0 --out {out}
display(Image(out))

## 9. Downstream classification

A classifier is trained on clear images of some areas and tested on other areas using the true
clear image (upper bound), the cloudy image (lower bound) and each reconstruction. Labels are
ESA WorldCover land-cover classes, **not crop types**. With only a few patches the numbers are
noisy; treat them as a demonstration of the pipeline.

In [ ]:
S = 'kharif_synth'
!python kharif/crop_classification.py --cache {BENCH} --split {S} --out {EVAL}/{S}_classification --preds full={EVAL}/{S}/pred_full.npy,{EVAL}/{S}/scale_full.npy matched={EVAL}/{S}/pred_matched_baseline.npy
!ls {EVAL}/{S}_classification

Everything is in `MyDrive/dieWolke/kharif_demo_eval`: `report.md` per split, per-image CSVs,
the PNG figures and the classification results. Download that folder and bring it back to update
the report.